In [ ]:
!pip install uv -q
!uv pip install "vllm==0.9.0" --torch-backend=cu128 --system -q
!pip install transformers==4.51.1 pyngrok openai -q
print("Installation done!")

In [ ]:
import vllm, torch
print(f"vLLM:    {vllm.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
import subprocess, time

log = open("/tmp/vllm.log", "w")

server = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-0.5B-Instruct",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--max-model-len", "2048",
        "--dtype", "half"
    ],
    stdout=log, stderr=log
)

print(f"Server starting (PID: {server.pid})...")
print("Wait for 3 minutes (for the model)")
time.sleep(180)

with open("/tmp/vllm.log", "r") as f:
    lines = f.read()
    if "Application startup complete" in lines:
        print("Server ready!")
    else:
        print("Still loading, check the logs.")
        print(lines[-2000:])

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

tunnel = ngrok.connect(8000)
VLLM_URL = tunnel.public_url

print(f"Server available on: {VLLM_URL}")
print(f"Copy this URL for app.py: {VLLM_URL}")